# OS Tutor: export the fine-tuned model for Ollama (no retraining)

Use this if the training notebook's session ended before its export step. You need **os-tutor-lora.zip**, which the
training notebook downloaded (or zip the files in your `qlora\os-tutor-lora` folder).

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. When the upload button appears in
section 1, choose `os-tutor-lora.zip`. At the end allow Google Drive access: the file is copied to *My Drive* as
`os-tutor.gguf` (~4.9 GB). About 25–30 minutes in total.
Then on your laptop: save it as `qlora\os-tutor.gguf` and run `qlora\setup_ollama_models.bat`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 1. Upload the adapter (os-tutor-lora.zip, or the files inside it)

In [ ]:
import os, shutil, zipfile
from google.colab import files
os.makedirs("os-tutor-lora", exist_ok=True)
for name in files.upload():
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall("os-tutor-lora")
    else:
        shutil.move(name, os.path.join("os-tutor-lora", name))
ADAPTER = next(root for root, _, fs in os.walk("os-tutor-lora") if "adapter_config.json" in fs)
print(ADAPTER, sorted(os.listdir(ADAPTER)))

## 2. Load Llama 3.1 8B with the trained adapter

In [ ]:
import json
from transformers import AutoTokenizer
base = json.load(open(os.path.join(ADAPTER, "adapter_config.json")))["base_model_name_or_path"]
print("base model:", base)
if not os.path.exists(os.path.join(ADAPTER, "tokenizer_config.json")):
    AutoTokenizer.from_pretrained(base).save_pretrained(ADAPTER)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER, max_seq_length=2048, load_in_4bit=True)

## 3. Quick check that it is the fine-tuned tutor

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = "You are OS Tutor, a friendly tutor for the VIT BCSE303 Operating Systems course. You teach CPU scheduling and process synchronization (critical sections, semaphores, classic problems, deadlocks, the Banker's algorithm). Explain from scratch: an everyday example first, then the exact rule. Follow the class-notes conventions: ties go to the earlier arrival; a lower priority number means higher priority unless the question says otherwise; in Round Robin a process arriving when a slice ends joins the queue before the preempted one; TAT = CT \u2212 AT, WT = TAT \u2212 BT, RT = first start \u2212 AT. When a student shows you their answer, find the first mistake and give a hint before giving the answer away. Keep replies focused, and end with one short question or next step unless the student only wanted a fact."
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Explain SRTF to me like I'm new to it"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=250, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 4. Merge and export (GGUF Q4_K_M, ~15–25 minutes)

In [ ]:
DO_GGUF = True
if DO_GGUF:
    model.save_pretrained_gguf("os-tutor-gguf", tokenizer, quantization_method="q4_k_m")
    print("export finished")

## 5. Copy the model file to Google Drive

In [ ]:
import os, shutil
def find_q4():
    hits = []
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in ("drive", "sample_data") and not d.startswith(".")]
        hits += [os.path.join(root, f) for f in files if f.lower().endswith(".gguf") and "q4_k_m" in f.lower()]
    return hits
hits = find_q4()
assert hits, "No Q4_K_M .gguf file found. Did the export cell above finish without an error?"
src = max(hits, key=os.path.getsize)
print("model file:", src, f"({os.path.getsize(src) / 1e9:.1f} GB)")
if os.path.exists("/kaggle/working"):
    shutil.copy(src, "/kaggle/working/os-tutor.gguf")
    print("Kaggle: download os-tutor.gguf from the Output panel on the right.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(src, "/content/drive/MyDrive/os-tutor.gguf")
    print("Copied to Google Drive -> My Drive -> os-tutor.gguf")
    print("Download it, save it on your laptop as qlora\\os-tutor.gguf and run qlora\\setup_ollama_models.bat")